# Base de Dados Financeira - Versão em Português

Este notebook carrega todas as tabelas da base de dados financeira do BIRD benchmark com os nomes das colunas traduzidos para português brasileiro.

## Requisitos

### 1. Download da Base de Dados

Faça o download da base de dados MiniDev do repositório oficial:
- **Repositório:** https://github.com/bird-bench/mini_dev
- **Arquivo necessário:** `financial.sqlite` localizado em `dev_databases/financial/`

### 2. Estrutura de Pastas Esperada

O notebook espera que o arquivo `financial.sqlite` esteja no seguinte caminho relativo:
```
baseMiniDev/minidev/MINIDEV/dev_databases/financial/financial.sqlite
```

### 3. Bibliotecas Python Necessárias

Instale as seguintes bibliotecas caso ainda não as tenha:

```bash
pip install pandas deep-translator
```

Notas:
- `sqlite3` geralmente já vem incluído na instalação padrão do Python.
- `deep-translator` é uma biblioteca robusta para tradução automática que suporta múltiplos serviços (Google Translate, Microsoft, etc.).

## Sobre a Base de Dados

Esta base de dados contém informações financeiras de um banco tcheco, incluindo:
- **account**: Contas bancárias
- **card**: Cartões de crédito
- **client**: Clientes do banco
- **disp**: Disposições (relacionamento cliente-conta)
- **district**: Informações demográficas dos distritos
- **loan**: Empréstimos concedidos

- **order**: Ordens de pagamento permanentes- **trans**: Transações bancárias

## Importação de Bibliotecas

In [2]:
import sqlite3
import pandas as pd
import os
from pathlib import Path
from deep_translator import GoogleTranslator
import unicodedata

## Função de Tradução Automática

Utilizamos a biblioteca `deep-translator` para traduzir automaticamente os valores das colunas do tcheco para o português.

In [3]:
# Inicializar tradutor com detecção automática para português
tradutor = GoogleTranslator(source='auto', target='pt')

def remover_acentos(texto):
    """
    Remove acentos e caracteres especiais de um texto.
    
    Parâmetros:
    - texto: string para normalizar
    
    Retorna:
    - string sem acentos
    """
    if pd.isna(texto):
        return texto
    # Normaliza o texto (NFD = Normalization Form Decomposed)
    # e remove marcas diacríticas (acentos)
    nfkd = unicodedata.normalize('NFD', str(texto))
    texto_sem_acento = ''.join([c for c in nfkd if not unicodedata.combining(c)])
    return texto_sem_acento

def traduzir_valores_coluna(serie, valores_unicos_limite=50):
    """
    Traduz os valores únicos de uma Series do pandas usando deep-translator.
    Remove acentos e caracteres especiais das traduções.
    
    Parâmetros:
    - serie: pd.Series com valores a serem traduzidos
    - valores_unicos_limite: limite de valores únicos para traduzir (segurança)
    
    Retorna:
    - pd.Series com valores traduzidos e sem acentos
    """
    # Obter valores únicos não nulos
    valores_unicos = serie.dropna().unique()
    
    # Verificar se o número de valores únicos é razoável
    if len(valores_unicos) > valores_unicos_limite:
        print(f"Aviso: Muitos valores únicos ({len(valores_unicos)}). Pulando tradução.")
        return serie
    
    # Criar dicionário de tradução
    dicionario_traducao = {}
    for valor in valores_unicos:
        try:
            # Traduzir o valor
            traducao = tradutor.translate(str(valor))
            # Remover acentos e converter para maiúsculas
            traducao_sem_acento = remover_acentos(traducao).upper()
            dicionario_traducao[valor] = traducao_sem_acento
        except Exception as e:
            print(f"Erro ao traduzir '{valor}': {e}")
            dicionario_traducao[valor] = valor  # Manter original em caso de erro
    
    # Aplicar tradução
    return serie.replace(dicionario_traducao)

print("Função de tradução automática configurada com sucesso.")

Função de tradução automática configurada com sucesso.


## Configuração do Caminho da Base de Dados

In [4]:

def detect_repo_root(start: Path) -> Path:
    env_root = os.getenv("TEXT2SQL_EVAL_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "pyproject.toml").exists() and (candidate / "config/config.yaml").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "config/config.yaml").exists():
            return candidate

    sibling = start / "text2sql-eval"
    if (sibling / "pyproject.toml").exists() and (sibling / "config/config.yaml").exists():
        return sibling

    raise RuntimeError(
        f"Could not detect repo root from {start}. "
        "Set TEXT2SQL_EVAL_ROOT=/absolute/path/to/text2sql-eval"
    )

REPO_ROOT = detect_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

CONFIG_PATH = REPO_ROOT / "config/config.yaml"

print("repo:", REPO_ROOT)


repo: /home/mileto/dev/text2sql-eval


In [5]:
# Defina o caminho para o arquivo SQLite
#db_path = Path('baseMiniDev/minidev/MINIDEV/dev_databases/financial/financial.sqlite')
# db_path = Path('./mini-dev/original/financial.sqlite')
db_path = Path('./data/database.sqlite')
# Verificar se o arquivo existe
if not db_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {db_path}\n"
        "Por favor, faça o download da base de dados de: https://github.com/bird-bench/mini_dev"
    )

# Criar conexão com o banco de dados
conn = sqlite3.connect(db_path)
print(f"Conexão estabelecida com: {db_path}")

Conexão estabelecida com: data/database.sqlite


## Função de Tradução de Nomes de Colunas

Função auxiliar para traduzir automaticamente os nomes das colunas do inglês para o português.

In [89]:
# Mapeamento manual e definitivo dos nomes das colunas baseado no database_description
dicionario_colunas_por_tabela = {
    "account": {
        "account_id": "id_conta",
        "district_id": "id_distrito",
        "frequency": "frequencia",
        "date": "data"
    },
    "card": {
        "card_id": "id_cartao",
        "disp_id": "id_disposicao",
        "type": "tipo",
        "issued": "emitido"
    },
    "client": {
        "client_id": "id_cliente",
        "gender": "genero",
        "birth_date": "data_nascimento",
        "district_id": "id_distrito"
    },
    "disp": {
        "disp_id": "id_disposicao",
        "client_id": "id_cliente",
        "account_id": "id_conta",
        "type": "tipo"
    },
    "district": {
        "district_id": "id_distrito",
        "A2": "A2",
        "A3": "A3",
        "A4": "A4",
        "A5": "A5",
        "A6": "A6",
        "A7": "A7",
        "A8": "A8",
        "A9": "A9",
        "A10":"A10",
        "A11":"A11",
        "A12":"A12",
        "A13":"A13",
        "A14":"A14",
        "A15":"A15",
        "A16":"A16",
    },
    "loan": {
        "loan_id": "id_emprestimo",
        "account_id": "id_conta",
        "date": "data",
        "amount": "valor",
        "duration": "duracao",
        "payments": "parcelas",
        "status": "status"
    },
    "order": {
        "order_id": "id_ordem",
        "account_id": "id_conta",
        "bank_to": "banco_para",
        "account_to": "conta_para",
        "amount": "valor",
        "k_symbol": "simbolo_k"
    },
    "trans": {
        "trans_id": "id_transacao",
        "account_id": "id_conta",
        "date": "data",
        "type": "tipo",
        "operation": "operacao",
        "amount": "valor",
        "balance": "saldo",
        "k_symbol": "simbolo_k",
        "bank": "banco",
        "account": "conta"
    }
}

def traduzir_nomes_colunas(df, table_name):
    """
    Traduz os nomes das colunas de um DataFrame usando nosso dicionário mapeado.
    """
    if table_name in dicionario_colunas_por_tabela:
        return df.rename(columns=dicionario_colunas_por_tabela[table_name])
    return df

print('Função de tradução de colunas atualizada com sucesso para dicionário estático.')


Função de tradução de colunas atualizada com sucesso para dicionário estático.


# Tabela 1: Account (Contas Bancárias)

Contém informações sobre as contas bancárias, incluindo localização da agência, periodicidade de emissão de extratos e data de abertura.

In [90]:
# Importar tabela account
df_conta = pd.read_sql_query("SELECT * FROM account", conn)
df_conta = traduzir_nomes_colunas(df_conta, 'account')

print(f"Tabela 'Account' carregada: {df_conta.shape[0]} registros, {df_conta.shape[1]} colunas")
print(f"Colunas: {', '.join(df_conta.columns)}")

Tabela 'Account' carregada: 4500 registros, 4 colunas
Colunas: id_conta, id_distrito, frequencia, data


## Visualização ANTES da tradução

In [91]:
# Visualização ANTES da tradução
colunas_freq = [col for col in df_conta.columns if 'freq' in col.lower() or 'period' in col.lower()]
if colunas_freq:
    coluna_frequencia = colunas_freq[0]
    print(f"Valores ORIGINAIS na coluna '{coluna_frequencia}':")
    print(df_conta[coluna_frequencia].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_conta.head()

Valores ORIGINAIS na coluna 'frequencia':
frequencia
POPLATEK MESICNE      4167
POPLATEK TYDNE         240
POPLATEK PO OBRATU      93
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,id_conta,id_distrito,frequencia,data
0,1,18,POPLATEK MESICNE,1995-03-24
1,2,1,POPLATEK MESICNE,1993-02-26
2,3,5,POPLATEK MESICNE,1997-07-07
3,4,12,POPLATEK MESICNE,1996-02-21
4,5,15,POPLATEK MESICNE,1997-05-30


## Tradução dos valores categóricos

In [92]:
# Traduzir valores categóricos
dicionario_conta_frequencia = {
    'POPLATEK MESICNE': 'TAXA MENSAL',
    'POPLATEK TYDNE': 'TAXA SEMANAL',
    'POPLATEK PO OBRATU': 'TAXA POR OPERACAO'
}

colunas_freq = [col for col in df_conta.columns if 'freq' in col.lower() or 'period' in col.lower()]
if colunas_freq:
    coluna_frequencia = colunas_freq[0]
    df_conta[coluna_frequencia] = df_conta[coluna_frequencia].replace(dicionario_conta_frequencia)
    print(f"Valores TRADUZIDOS na coluna '{coluna_frequencia}':")
    print(df_conta[coluna_frequencia].value_counts())
else:
    print("Coluna de frequência não encontrada.")


Valores TRADUZIDOS na coluna 'frequencia':
frequencia
TAXA MENSAL          4167
TAXA SEMANAL          240
TAXA POR OPERACAO      93
Name: count, dtype: int64


## Visualização DEPOIS da tradução

In [93]:
print("Primeiras linhas DEPOIS da tradução:")
df_conta.head()

Primeiras linhas DEPOIS da tradução:


,id_conta,id_distrito,frequencia,data
0,1,18,TAXA MENSAL,1995-03-24
1,2,1,TAXA MENSAL,1993-02-26
2,3,5,TAXA MENSAL,1997-07-07
3,4,12,TAXA MENSAL,1996-02-21
4,5,15,TAXA MENSAL,1997-05-30


# Tabela 2: Card (Cartões de Crédito)

Informações sobre cartões de crédito emitidos, incluindo tipo (junior, classic, gold) e data de emissão.

In [94]:
# Importar tabela card
df_cartao = pd.read_sql_query("SELECT * FROM card", conn)
df_cartao = traduzir_nomes_colunas(df_cartao, 'card')

print(f"Tabela 'Card' carregada: {df_cartao.shape[0]} registros, {df_cartao.shape[1]} colunas")
print(f"Colunas: {', '.join(df_cartao.columns)}")

Tabela 'Card' carregada: 892 registros, 4 colunas
Colunas: id_cartao, id_disposicao, tipo, emitido


In [95]:
df_cartao.head()

,id_cartao,id_disposicao,tipo,emitido
0,1,9,gold,1998-10-16
1,2,19,classic,1998-03-13
2,3,41,gold,1995-09-03
3,4,42,classic,1998-11-26
4,5,51,junior,1995-04-24


# Tabela 3: Client (Clientes)

Dados cadastrais dos clientes do banco, incluindo informações demográficas básicas.

In [96]:
# Importar tabela client
df_cliente = pd.read_sql_query("SELECT * FROM client", conn)
df_cliente = traduzir_nomes_colunas(df_cliente, 'client')

print(f"Tabela 'Client' carregada: {df_cliente.shape[0]} registros, {df_cliente.shape[1]} colunas")
print(f"Colunas: {', '.join(df_cliente.columns)}")


Tabela 'Client' carregada: 5369 registros, 4 colunas
Colunas: id_cliente, genero, data_nascimento, id_distrito


# Tabela 4: Disp (Disposições)

Relacionamento entre clientes e contas, indicando o tipo de disposição (proprietário, usuário ou disponente).

In [97]:
# Importar tabela disp
df_disposicao = pd.read_sql_query("SELECT * FROM disp", conn)
df_disposicao = traduzir_nomes_colunas(df_disposicao, 'disp')

print(f"Tabela 'Disp' carregada: {df_disposicao.shape[0]} registros, {df_disposicao.shape[1]} colunas")


Tabela 'Disp' carregada: 5369 registros, 4 colunas


## Visualização ANTES da tradução

In [98]:
# Visualização ANTES da tradução
colunas_tipo = [col for col in df_disposicao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
if colunas_tipo:
    coluna_tipo = colunas_tipo[0]
    print(f"Valores ORIGINAIS na coluna '{coluna_tipo}':")
    print(df_disposicao[coluna_tipo].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_disposicao.head()

Valores ORIGINAIS na coluna 'tipo':
tipo
OWNER        4500
DISPONENT     869
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,id_disposicao,id_cliente,id_conta,tipo
0,1,1,1,OWNER
1,2,2,2,OWNER
2,3,3,2,DISPONENT
3,4,4,3,OWNER
4,5,5,3,DISPONENT


## Tradução dos valores categóricos

In [99]:
# Traduzir valores categóricos
dicionario_disp_tipo = {
    'OWNER': 'TITULAR',
    'DISPONENT': 'AUTORIZADO'
}

colunas_tipo = [col for col in df_disposicao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
if colunas_tipo:
    coluna_tipo = colunas_tipo[0]
    df_disposicao[coluna_tipo] = df_disposicao[coluna_tipo].replace(dicionario_disp_tipo)
    print(f"Valores TRADUZIDOS na coluna '{coluna_tipo}':")
    print(df_disposicao[coluna_tipo].value_counts())
else:
    print("Coluna de tipo não encontrada.")


Valores TRADUZIDOS na coluna 'tipo':
tipo
TITULAR       4500
AUTORIZADO     869
Name: count, dtype: int64


## Visualização DEPOIS da tradução

In [100]:
print("Primeiras linhas DEPOIS da tradução:")
df_disposicao.head()

Primeiras linhas DEPOIS da tradução:


,id_disposicao,id_cliente,id_conta,tipo
0,1,1,1,TITULAR
1,2,2,2,TITULAR
2,3,3,2,AUTORIZADO
3,4,4,3,TITULAR
4,5,5,3,AUTORIZADO


# Tabela 5: District (Distritos)

Informações demográficas e socioeconômicas dos distritos, incluindo dados populacionais, taxa de desemprego, salário médio e criminalidade.

In [101]:
# Importar tabela district
df_distrito = pd.read_sql_query("SELECT * FROM district", conn)
df_distrito = traduzir_nomes_colunas(df_distrito, 'district')

print(f"Tabela 'District' carregada: {df_distrito.shape[0]} registros, {df_distrito.shape[1]} colunas")
print(f"Colunas: {', '.join(df_distrito.columns)}")
print("\nPrimeiras linhas da tabela District:")
print(df_distrito.head())


Tabela 'District' carregada: 77 registros, 16 colunas
Colunas: id_distrito, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11, A12, A13, A14, A15, A16

Primeiras linhas da tabela District:
   id_distrito           A2               A3       A4  A5  A6 A7  A8  A9  \
0            1  Hl.m. Praha           Prague  1204953   0   0  0   1   1   
1            2      Benesov  central Bohemia    88884  80  26  6   2   5   
2            3       Beroun  central Bohemia    75232  55  26  4   1   5   
3            4       Kladno  central Bohemia   149893  63  29  6   2   6   
4            5        Kolin  central Bohemia    95616  65  30  4   1   6   

     A10    A11  A12   A13  A14      A15    A16  
0  100.0  12541  0.2  0.43  167  85677.0  99107  
1   46.7   8507  1.6  1.85  132   2159.0   2674  
2   41.7   8980  1.9  2.21  111   2824.0   2813  
3   67.4   9753  4.6  5.05  109   5244.0   5892  
4   51.4   9307  3.8  4.43  118   2616.0   3040  


# Tabela 6: Loan (Empréstimos)

Registro de empréstimos concedidos, incluindo valor, prazo, parcelas mensais e status de pagamento.

In [102]:
# Importar tabela loan
df_emprestimo = pd.read_sql_query("SELECT * FROM loan", conn)
df_emprestimo = traduzir_nomes_colunas(df_emprestimo, 'loan')

print(f"Tabela 'Loan' carregada: {df_emprestimo.shape[0]} registros, {df_emprestimo.shape[1]} colunas")
print(f"Colunas: {', '.join(df_emprestimo.columns)}")
print(df_emprestimo.head())


Tabela 'Loan' carregada: 682 registros, 7 colunas
Colunas: id_emprestimo, id_conta, data, valor, duracao, parcelas, status
   id_emprestimo  id_conta        data   valor  duracao  parcelas status
0           4959         2  1994-01-05   80952       24    3373.0      A
1           4961        19  1996-04-29   30276       12    2523.0      B
2           4962        25  1997-12-08   30276       12    2523.0      A
3           4967        37  1998-10-14  318480       60    5308.0      D
4           4968        38  1998-04-19  110736       48    2307.0      C


# Tabela 7: Order (Ordens de Pagamento)

Ordens de pagamento permanentes configuradas nas contas, especificando beneficiário, valor e finalidade.

In [103]:
# Importar tabela order
df_ordem = pd.read_sql_query("SELECT * FROM `order`", conn)
df_ordem = traduzir_nomes_colunas(df_ordem, 'order')

# Substituir códigos conhecidos por descrições em inglês
# para depois traduzir automaticamente para português.
colunas_para_substituicao = [
    col for col in df_ordem.columns
    if ('value' in col.lower() and 'description' in col.lower())
    or ('valor' in col.lower() and 'descricao' in col.lower())
    or ('simbolo' in col.lower() or 'symbol' in col.lower())
]

if colunas_para_substituicao:
    mapeamento_codigos_order = {
        'POJISTNE': 'insurance payment',
        'SIPO': 'household payment',
        'LEASING': 'leasing',
        'UVER': 'loan payment'
    }
    for coluna in colunas_para_substituicao:
        df_ordem[coluna] = df_ordem[coluna].replace(mapeamento_codigos_order)
    print(f"Substituições aplicadas nas colunas: {', '.join(colunas_para_substituicao)}")
else:
    print("Nenhuma coluna alvo encontrada para substituição de códigos da tabela Order.")

print(f"Tabela 'Order' carregada: {df_ordem.shape[0]} registros, {df_ordem.shape[1]} colunas")


Substituições aplicadas nas colunas: simbolo_k
Tabela 'Order' carregada: 6471 registros, 6 colunas


In [104]:
df_ordem

,id_ordem,id_conta,banco_para,conta_para,valor,simbolo_k
0,29401,1,YZ,87144583,2452.0,household payment
1,29402,2,ST,89597016,3372.7,loan payment
2,29403,2,QR,13943797,7266.0,household payment
3,29404,3,WX,83084338,1135.0,household payment
4,29405,3,CD,24485939,327.0,
...,...,...,...,...,...,...
6466,46334,11362,YZ,70641225,4780.0,household payment
6467,46335,11362,MN,78507822,56.0,
6468,46336,11362,ST,40799850,330.0,insurance payment
6469,46337,11362,KL,20009470,129.0,


## Visualização ANTES da tradução

In [105]:
# Visualização ANTES da tradução
colunas_simbolo = [col for col in df_ordem.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]
if colunas_simbolo:
    coluna_simbolo = colunas_simbolo[0]
    print(f"Valores ORIGINAIS na coluna '{coluna_simbolo}':")
    print(df_ordem[coluna_simbolo].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_ordem.head()

Valores ORIGINAIS na coluna 'simbolo_k':
simbolo_k
household payment    3502
                     1379
loan payment          717
insurance payment     532
leasing               341
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,id_ordem,id_conta,banco_para,conta_para,valor,simbolo_k
0,29401,1,YZ,87144583,2452.0,household payment
1,29402,2,ST,89597016,3372.7,loan payment
2,29403,2,QR,13943797,7266.0,household payment
3,29404,3,WX,83084338,1135.0,household payment
4,29405,3,CD,24485939,327.0,


## Tradução dos valores categóricos

In [106]:
# Traduzir valores categóricos
dicionario_ordem_simbolo = {
    'household payment': 'CONTAS DE CONSUMO',
    'loan payment': 'PAGAMENTO DE EMPRESTIMO',
    'insurance payment': 'PAGAMENTO DE SEGURO',
    'leasing': 'LEASING',
    'POJISTNE': 'PAGAMENTO DE SEGURO',
    'SIPO': 'CONTAS DE CONSUMO',
    'LEASING': 'LEASING',
    'UVER': 'PAGAMENTO DE EMPRESTIMO',
    '': ''
}

colunas_simbolo = [col for col in df_ordem.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]
colunas_value_description = [
    col for col in df_ordem.columns
    if ('value' in col.lower() and 'description' in col.lower())
    or ('valor' in col.lower() and 'descricao' in col.lower())
]

if colunas_simbolo:
    coluna_simbolo = colunas_simbolo[0]
    df_ordem[coluna_simbolo] = df_ordem[coluna_simbolo].replace(dicionario_ordem_simbolo)
    print(f"Valores TRADUZIDOS na coluna '{coluna_simbolo}':")
    print(df_ordem[coluna_simbolo].value_counts())
else:
    print("Coluna símbolo não encontrada.")

if colunas_value_description:
    coluna_value_description = colunas_value_description[0]
    df_ordem[coluna_value_description] = df_ordem[coluna_value_description].replace(dicionario_ordem_simbolo)
    print(f"Valores TRADUZIDOS na coluna '{coluna_value_description}':")
    print(df_ordem[coluna_value_description].value_counts())
else:
    print("Coluna value_description (ou equivalente traduzida) não encontrada para tradução.")


Valores TRADUZIDOS na coluna 'simbolo_k':
simbolo_k
CONTAS DE CONSUMO          3502
                           1379
PAGAMENTO DE EMPRESTIMO     717
PAGAMENTO DE SEGURO         532
LEASING                     341
Name: count, dtype: int64
Coluna value_description (ou equivalente traduzida) não encontrada para tradução.


## Visualização DEPOIS da tradução

In [107]:
print("Primeiras linhas DEPOIS da tradução:")
df_ordem.head()

Primeiras linhas DEPOIS da tradução:


,id_ordem,id_conta,banco_para,conta_para,valor,simbolo_k
0,29401,1,YZ,87144583,2452.0,CONTAS DE CONSUMO
1,29402,2,ST,89597016,3372.7,PAGAMENTO DE EMPRESTIMO
2,29403,2,QR,13943797,7266.0,CONTAS DE CONSUMO
3,29404,3,WX,83084338,1135.0,CONTAS DE CONSUMO
4,29405,3,CD,24485939,327.0,


# Tabela 8: Trans (Transações)

Registro detalhado de todas as transações bancárias, incluindo depósitos, saques, transferências e saldo resultante.

In [108]:
# Importar tabela trans
df_transacao = pd.read_sql_query("SELECT * FROM trans", conn)
df_transacao = traduzir_nomes_colunas(df_transacao, 'trans')

print(f"Tabela 'Trans' carregada: {df_transacao.shape[0]} registros, {df_transacao.shape[1]} colunas")


Tabela 'Trans' carregada: 1056320 registros, 10 colunas


## Visualização ANTES da tradução

In [109]:
# Visualização ANTES da tradução - amostra limitada devido ao tamanho da tabela
colunas_tipo = [col for col in df_transacao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
colunas_operacao = [col for col in df_transacao.columns if 'operacao' in col.lower() or 'operation' in col.lower()]
colunas_simbolo = [col for col in df_transacao.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]

if colunas_tipo:
    print(f"Valores ORIGINAIS na coluna '{colunas_tipo[0]}':")
    print(df_transacao[colunas_tipo[0]].value_counts())
    print()

if colunas_operacao:
    print(f"Valores ORIGINAIS na coluna '{colunas_operacao[0]}':")
    print(df_transacao[colunas_operacao[0]].value_counts())
    print()

if colunas_simbolo:
    print(f"Valores ORIGINAIS na coluna '{colunas_simbolo[0]}':")
    print(df_transacao[colunas_simbolo[0]].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_transacao.head()

Valores ORIGINAIS na coluna 'tipo':
tipo
VYDAJ     634571
PRIJEM    405083
VYBER      16666
Name: count, dtype: int64

Valores ORIGINAIS na coluna 'operacao':
operacao
VYBER             434918
PREVOD NA UCET    208283
VKLAD             156743
PREVOD Z UCTU      65226
VYBER KARTOU        8036
Name: count, dtype: int64

Valores ORIGINAIS na coluna 'simbolo_k':
simbolo_k
UROK           183114
SLUZBY         155832
SIPO           118065
                53433
DUCHOD          30338
POJISTNE        18500
UVER            13580
SANKC. UROK      1577
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,id_transacao,id_conta,data,tipo,operacao,valor,saldo,simbolo_k,banco,conta
0,1,1,1995-03-24,PRIJEM,VKLAD,1000,1000,NaN,NaN,NaN
1,5,1,1995-04-13,PRIJEM,PREVOD Z UCTU,3679,4679,NaN,AB,41403269.0
2,6,1,1995-05-13,PRIJEM,PREVOD Z UCTU,3679,20977,NaN,AB,41403269.0
3,7,1,1995-06-13,PRIJEM,PREVOD Z UCTU,3679,26835,NaN,AB,41403269.0
4,8,1,1995-07-13,PRIJEM,PREVOD Z UCTU,3679,30415,NaN,AB,41403269.0


## Tradução dos valores categóricos

In [110]:
# Traduzir valores das colunas categóricas
dicionario_trans_tipo = {
    'PRIJEM': 'CREDITO',
    'VYDAJ': 'DEBITO',
    'VYBER': 'SAQUE'
}

dicionario_trans_operacao = {
    'VYBER': 'SAQUE EM DINHEIRO',
    'PREVOD NA UCET': 'TRANSFERENCIA ENVIADA',
    'VKLAD': 'DEPOSITO EM DINHEIRO',
    'PREVOD Z UCTU': 'TRANSFERENCIA RECEBIDA',
    'VYBER KARTOU': 'SAQUE NO CARTAO'
}

dicionario_trans_simbolo = {
    'UROK': 'JUROS CREDITADOS',
    'SLUZBY': 'TAXA DE SERVICO',
    'SIPO': 'CONTAS DE CONSUMO',
    'DUCHOD': 'PENSAO',
    'POJISTNE': 'PAGAMENTO DE SEGURO',
    'UVER': 'PAGAMENTO DE EMPRESTIMO',
    'SANKC. UROK': 'JUROS DE MORA',
    '': '',
    ' ': ' '
}

colunas_tipo = [col for col in df_transacao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
colunas_operacao = [col for col in df_transacao.columns if 'operacao' in col.lower() or 'operation' in col.lower()]
colunas_simbolo = [col for col in df_transacao.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]

if colunas_tipo:
    coluna_tipo = colunas_tipo[0]
    df_transacao[coluna_tipo] = df_transacao[coluna_tipo].replace(dicionario_trans_tipo)
    print(f"Valores TRADUZIDOS na coluna '{coluna_tipo}':")
    print(df_transacao[coluna_tipo].value_counts())
    print()

if colunas_operacao:
    coluna_operacao = colunas_operacao[0]
    df_transacao[coluna_operacao] = df_transacao[coluna_operacao].replace(dicionario_trans_operacao)
    print(f"Valores TRADUZIDOS na coluna '{coluna_operacao}':")
    print(df_transacao[coluna_operacao].value_counts())
    print()

if colunas_simbolo:
    coluna_simbolo = colunas_simbolo[0]
    df_transacao[coluna_simbolo] = df_transacao[coluna_simbolo].replace(dicionario_trans_simbolo)
    print(f"Valores TRADUZIDOS na coluna '{coluna_simbolo}':")
    print(df_transacao[coluna_simbolo].value_counts())


Valores TRADUZIDOS na coluna 'tipo':
tipo
DEBITO     634571
CREDITO    405083
SAQUE       16666
Name: count, dtype: int64

Valores TRADUZIDOS na coluna 'operacao':
operacao
SAQUE EM DINHEIRO         434918
TRANSFERENCIA ENVIADA     208283
DEPOSITO EM DINHEIRO      156743
TRANSFERENCIA RECEBIDA     65226
SAQUE NO CARTAO             8036
Name: count, dtype: int64

Valores TRADUZIDOS na coluna 'simbolo_k':
simbolo_k
JUROS CREDITADOS           183114
TAXA DE SERVICO            155832
CONTAS DE CONSUMO          118065
                            53433
PENSAO                      30338
PAGAMENTO DE SEGURO         18500
PAGAMENTO DE EMPRESTIMO     13580
JUROS DE MORA                1577
Name: count, dtype: int64


## Visualização DEPOIS da tradução

In [111]:
print("Primeiras linhas DEPOIS da tradução:")
df_transacao.head()

Primeiras linhas DEPOIS da tradução:


,id_transacao,id_conta,data,tipo,operacao,valor,saldo,simbolo_k,banco,conta
0,1,1,1995-03-24,CREDITO,DEPOSITO EM DINHEIRO,1000,1000,NaN,NaN,NaN
1,5,1,1995-04-13,CREDITO,TRANSFERENCIA RECEBIDA,3679,4679,NaN,AB,41403269.0
2,6,1,1995-05-13,CREDITO,TRANSFERENCIA RECEBIDA,3679,20977,NaN,AB,41403269.0
3,7,1,1995-06-13,CREDITO,TRANSFERENCIA RECEBIDA,3679,26835,NaN,AB,41403269.0
4,8,1,1995-07-13,CREDITO,TRANSFERENCIA RECEBIDA,3679,30415,NaN,AB,41403269.0


# Resumo Final

Todas as tabelas foram carregadas e traduzidas com sucesso!

In [112]:
# Resumo de todas as tabelas carregadas
tabelas = {
    'df_conta': df_conta,
    'df_cartao': df_cartao,
    'df_cliente': df_cliente,
    'df_disposicao': df_disposicao,
    'df_distrito': df_distrito,
    'df_emprestimo': df_emprestimo,
    'df_ordem': df_ordem,
    'df_transacao': df_transacao
}

print("\nResumo das Tabelas Carregadas:")
print("=" * 80)
for nome_df, df in tabelas.items():
    print(f"{nome_df:20} | Registros: {df.shape[0]:6} | Colunas: {df.shape[1]:2}")
print("=" * 80)


Resumo das Tabelas Carregadas:
df_conta             | Registros:   4500 | Colunas:  4
df_cartao            | Registros:    892 | Colunas:  4
df_cliente           | Registros:   5369 | Colunas:  4
df_disposicao        | Registros:   5369 | Colunas:  4
df_distrito          | Registros:     77 | Colunas: 16
df_emprestimo        | Registros:    682 | Colunas:  7
df_ordem             | Registros:   6471 | Colunas:  6
df_transacao         | Registros: 1056320 | Colunas: 10


## Persistir as tabelas traduzidas no SQLite

A célula abaixo cria um novo banco SQLite com tabelas e colunas em português a partir dos `DataFrame`s traduzidos.

Por segurança, o comportamento padrão é gravar um novo arquivo `financial_traduzido.sqlite` dentro de `mini-dev/pt-br/`.
Se quiser substituir `financial.sqlite`, altere `substituir_arquivo_original = True`. Nesse caso, o notebook cria um backup automático antes da troca.

In [113]:
# Persistir as tabelas traduzidas no SQLite
from shutil import copy2

tabelas_traduzidas = [
    ("account", "conta", df_conta),
    ("card", "cartao", df_cartao),
    ("client", "cliente", df_cliente),
    ("disp", "disp", df_disposicao),
    ("district", "distrito", df_distrito),
    ("loan", "emprestimo", df_emprestimo),
    ("order", "ordem", df_ordem),
    ("trans", "trans", df_transacao),
]

substituir_arquivo_original = False
db_path_destino = (
    db_path
    if substituir_arquivo_original
    else db_path.with_name("financial_traduzido.sqlite")
)

def quote_ident(nome):
    return '"' + str(nome).replace('"', '""') + '"'

def table_info(connection, table_name):
    return connection.execute(
        f"PRAGMA table_info({quote_ident(table_name)})"
    ).fetchall()

def foreign_keys(connection, table_name):
    return connection.execute(
        f"PRAGMA foreign_key_list({quote_ident(table_name)})"
    ).fetchall()

schema_original = {
    tabela_origem: {
        "columns": table_info(conn, tabela_origem),
        "foreign_keys": foreign_keys(conn, tabela_origem),
    }
    for tabela_origem, _, _ in tabelas_traduzidas
}

mapeamento_tabelas = {
    tabela_origem: tabela_destino
    for tabela_origem, tabela_destino, _ in tabelas_traduzidas
}
mapeamento_colunas = {}

for tabela_origem, _, df in tabelas_traduzidas:
    colunas_originais = [row[1] for row in schema_original[tabela_origem]["columns"]]
    colunas_traduzidas = list(df.columns)
    if len(colunas_originais) != len(colunas_traduzidas):
        raise ValueError(
            f"Quantidade de colunas divergente em {tabela_origem}: "
            f"{len(colunas_originais)} no SQLite original e "
            f"{len(colunas_traduzidas)} no DataFrame traduzido."
        )
    mapeamento_colunas[tabela_origem] = dict(zip(colunas_originais, colunas_traduzidas))

temp_path = db_path_destino.with_name(f"{db_path_destino.stem}_tmp.sqlite")
if temp_path.exists():
    temp_path.unlink()

conn.close()
conn = None

conn_destino = sqlite3.connect(temp_path)
conn_destino.execute("PRAGMA foreign_keys = OFF")

for tabela_origem, tabela_destino, df in tabelas_traduzidas:
    colunas_pragma = schema_original[tabela_origem]["columns"]
    foreign_keys_pragma = schema_original[tabela_origem]["foreign_keys"]
    pk_cols = []
    definicoes_colunas = []

    for row in colunas_pragma:
        nome_original = row[1]
        nome_traduzido = mapeamento_colunas[tabela_origem][nome_original]
        tipo_coluna = row[2] or "TEXT"
        not_null = bool(row[3])
        valor_padrao = row[4]
        pk_ordem = row[5]

        partes = [quote_ident(nome_traduzido), tipo_coluna]
        if valor_padrao is not None:
            partes.append(f"DEFAULT {valor_padrao}")
        if len([col for col in colunas_pragma if col[5]]) == 1 and pk_ordem:
            partes.append("PRIMARY KEY")
        if not_null:
            partes.append("NOT NULL")

        definicoes_colunas.append(" ".join(partes))
        if pk_ordem:
            pk_cols.append((pk_ordem, nome_traduzido))

    constraints = []
    if len(pk_cols) > 1:
        pk_cols_ordenadas = [
            quote_ident(nome)
            for _, nome in sorted(pk_cols, key=lambda item: item[0])
        ]
        constraints.append(
            f"PRIMARY KEY ({', '.join(pk_cols_ordenadas)})"
        )

    for fk in foreign_keys_pragma:
        tabela_referenciada = fk[2]
        coluna_origem = fk[3]
        coluna_destino = fk[4]
        constraints.append(
            "FOREIGN KEY "
            f"({quote_ident(mapeamento_colunas[tabela_origem][coluna_origem])}) "
            f"REFERENCES {quote_ident(mapeamento_tabelas[tabela_referenciada])} "
            f"({quote_ident(mapeamento_colunas[tabela_referenciada][coluna_destino])})"
        )

    create_sql = (
        f"CREATE TABLE {quote_ident(tabela_destino)} (\n    "
        + ",\n    ".join(definicoes_colunas + constraints)
        + "\n)"
    )
    conn_destino.execute(create_sql)
    df.to_sql(tabela_destino, conn_destino, if_exists="append", index=False)
    print(
        f"Tabela traduzida gravada: {tabela_destino} "
        f"({df.shape[0]} registros, {df.shape[1]} colunas)"
    )

conn_destino.commit()
conn_destino.close()

if substituir_arquivo_original:
    backup_path = db_path.with_name(f"{db_path.stem}_antes_da_traducao.sqlite")
    if not backup_path.exists():
        copy2(db_path, backup_path)
        print(f"Backup criado em: {backup_path}")
    temp_path.replace(db_path)
    print(f"Banco original substituido por: {db_path}")
else:
    if db_path_destino.exists():
        db_path_destino.unlink()
    temp_path.replace(db_path_destino)
    print(f"Banco traduzido criado em: {db_path_destino}")

Tabela traduzida gravada: conta (4500 registros, 4 colunas)
Tabela traduzida gravada: cartao (892 registros, 4 colunas)
Tabela traduzida gravada: cliente (5369 registros, 4 colunas)
Tabela traduzida gravada: disp (5369 registros, 4 colunas)
Tabela traduzida gravada: distrito (77 registros, 16 colunas)
Tabela traduzida gravada: emprestimo (682 registros, 7 colunas)
Tabela traduzida gravada: ordem (6471 registros, 6 colunas)
Tabela traduzida gravada: trans (1056320 registros, 10 colunas)
Banco traduzido criado em: mini-dev/original/financial_traduzido.sqlite


In [7]:
import sqlite3
import pandas as pd
import IPython.display as display

# Conectar ao banco de dados traduzido
conn_test = sqlite3.connect(db_path)
print(db_path)

queries = [
    (89, "SELECT COUNT(T2.id_conta) FROM distrito AS T1 INNER JOIN conta AS T2 ON T1.id_distrito = T2.id_distrito WHERE T1.A3 = 'east Bohemia' AND T2.frequencia = 'TAXA POR OPERACAO'"),
    (92, "SELECT COUNT(DISTINCT T2.id_distrito) FROM cliente AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito WHERE T1.genero = 'F' AND T2.A11 BETWEEN 6000 AND 10000"),
    (93, "SELECT COUNT(T1.id_cliente) FROM cliente AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito WHERE T1.genero = 'M' AND T2.A3 = 'north Bohemia' AND T2.A11 > 8000"),
    (94, "SELECT T1.id_conta , ( SELECT MAX(A11) - MIN(A11) FROM distrito ) FROM conta AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito INNER JOIN disp AS T3 ON T1.id_conta = T3.id_conta INNER JOIN cliente AS T4 ON T3.id_cliente = T4.id_cliente WHERE T2.id_distrito = ( SELECT id_distrito FROM cliente WHERE genero = 'F' ORDER BY data_nascimento ASC LIMIT 1 ) ORDER BY T2.A11 DESC LIMIT 1"),
    (95, "SELECT T1.id_conta FROM conta AS T1 INNER JOIN disp AS T2 ON T1.id_conta = T2.id_conta INNER JOIN cliente AS T3 ON T2.id_cliente = T3.id_cliente INNER JOIN distrito AS T4 on T4.id_distrito = T1.id_distrito WHERE T2.id_cliente = ( SELECT id_cliente FROM cliente ORDER BY data_nascimento DESC LIMIT 1) GROUP BY T4.A11, T1.id_conta"),
    (98, "SELECT T2.id_conta FROM emprestimo AS T1 INNER JOIN conta AS T2 ON T1.id_conta = T2.id_conta WHERE STRFTIME('%Y', T1.data) = '1997' AND T2.frequencia = 'TAXA SEMANAL' ORDER BY T1.valor LIMIT 1"),
    (99, "SELECT T1.id_conta FROM emprestimo AS T1 INNER JOIN conta AS T2 ON T1.id_conta = T2.id_conta WHERE STRFTIME('%Y', T2.data) = '1993' AND T1.duracao > 12 ORDER BY T1.valor DESC LIMIT 1"),
    (100, "SELECT COUNT(T2.id_cliente) FROM distrito AS T1 INNER JOIN cliente AS T2 ON T1.id_distrito = T2.id_distrito WHERE T2.genero = 'F' AND STRFTIME('%Y', T2.data_nascimento) < '1950' AND T1.A2 = 'Sokolov'"),
    (112, "SELECT T1.A2 FROM distrito AS T1 INNER JOIN cliente AS T2 ON T1.id_distrito = T2.id_distrito WHERE T2.data_nascimento = '1976-01-29' AND T2.genero = 'F'"),
    (115, "SELECT CAST(SUM(T1.genero = 'M') AS REAL) * 100 / COUNT(T1.id_cliente) FROM cliente AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito WHERE T2.A3 = 'south Bohemia' GROUP BY T2.A4 ORDER BY T2.A4 DESC LIMIT 1"),
    (116, "SELECT CAST((SUM(IIF(T3.data = '1998-12-27', T3.saldo, 0)) - SUM(IIF(T3.data = '1993-03-22', T3.saldo, 0))) AS REAL) * 100 / SUM(IIF(T3.data = '1993-03-22', T3.saldo, 0)) FROM emprestimo AS T1 INNER JOIN conta AS T2 ON T1.id_conta = T2.id_conta INNER JOIN trans AS T3 ON T3.id_conta = T2.id_conta WHERE T1.data = '1993-07-05'"),
    (117, "SELECT (CAST(SUM(CASE WHEN status = 'A' THEN valor ELSE 0 END) AS REAL) * 100) / SUM(valor) FROM emprestimo"),
    (118, "SELECT CAST(SUM(status = 'C') AS REAL) * 100 / COUNT(id_conta) FROM emprestimo WHERE valor < 100000"),
    (125, "SELECT CAST((T3.A13 - T3.A12) AS REAL) * 100 / T3.A12 FROM emprestimo AS T1 INNER JOIN conta AS T2 ON T1.id_conta = T2.id_conta INNER JOIN distrito AS T3 ON T2.id_distrito = T3.id_distrito WHERE T1.status = 'D'"),
    (128, "SELECT T2.A2, COUNT(T1.id_cliente) FROM cliente AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito WHERE T1.genero = 'F' GROUP BY T2.id_distrito, T2.A2 ORDER BY COUNT(T1.id_cliente) DESC LIMIT 9"),
    (136, "SELECT COUNT(T1.id_conta) FROM conta AS T1 INNER JOIN emprestimo AS T2 ON T1.id_conta = T2.id_conta WHERE T2.data BETWEEN '1995-01-01' AND '1997-12-31' AND T1.frequencia = 'TAXA MENSAL' AND T2.valor >= 250000"),
    (129, "SELECT DISTINCT T1.A2 FROM distrito AS T1 INNER JOIN conta AS T2 ON T1.id_distrito = T2.id_distrito INNER JOIN trans AS T3 ON T2.id_conta = T3.id_conta WHERE T3.tipo = 'DEBITO' AND T3.data LIKE '1996-01%' ORDER BY T1.A2 ASC LIMIT 10"),
    (137, "SELECT COUNT(T1.id_conta) FROM conta AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito INNER JOIN emprestimo AS T3 ON T1.id_conta = T3.id_conta WHERE T1.id_distrito = 1 AND (T3.status = 'C' OR T3.status = 'D')"),
    (138, "SELECT COUNT(T1.id_cliente) FROM cliente AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito WHERE T1.genero = 'M' AND T2.A15 = (SELECT T3.A15 FROM distrito AS T3 ORDER BY T3.A15 DESC LIMIT 1, 1)"),
    (145, "SELECT T1.id_conta FROM trans AS T1 INNER JOIN conta AS T2 ON T1.id_conta = T2.id_conta WHERE STRFTIME('%Y', T1.data) = '1998' AND T1.operacao = 'SAQUE NO CARTAO' AND T1.valor < (SELECT AVG(valor) FROM trans WHERE STRFTIME('%Y', data) = '1998')"),
    (149, "SELECT T3.tipo FROM distrito AS T1 INNER JOIN conta AS T2 ON T1.id_distrito = T2.id_distrito INNER JOIN disp AS T3 ON T2.id_conta = T3.id_conta WHERE T3.tipo != 'TITULAR' AND T1.A11 BETWEEN 8000 AND 9000"),
    (152, "SELECT AVG(T1.A15) FROM distrito AS T1 INNER JOIN conta AS T2 ON T1.id_distrito = T2.id_distrito WHERE STRFTIME('%Y', T2.data) >= '1997' AND T1.A15 > 4000"),
    (159, "SELECT T4.id_transacao FROM cliente AS T1 INNER JOIN disp AS T2 ON T1.id_cliente = T2.id_cliente INNER JOIN conta AS T3 ON T2.id_conta = T3.id_conta INNER JOIN trans AS T4 ON T3.id_conta = T4.id_conta WHERE T1.id_cliente = 3356 AND T4.operacao = 'SAQUE EM DINHEIRO'"),
    (168, "SELECT CAST(SUM(T2.genero = 'F') AS REAL) * 100 / COUNT(T2.id_cliente) FROM distrito AS T1 INNER JOIN cliente AS T2 ON T1.id_distrito = T2.id_distrito WHERE T1.A11 > 10000"),
    (169, "SELECT CAST((SUM(CASE WHEN STRFTIME('%Y', T1.data) = '1997' THEN T1.valor ELSE 0 END) - SUM(CASE WHEN STRFTIME('%Y', T1.data) = '1996' THEN T1.valor ELSE 0 END)) AS REAL) * 100 / SUM(CASE WHEN STRFTIME('%Y', T1.data) = '1996' THEN T1.valor ELSE 0 END) FROM emprestimo AS T1 INNER JOIN conta AS T2 ON T1.id_conta = T2.id_conta INNER JOIN disp AS T3 ON T3.id_conta = T2.id_conta INNER JOIN cliente AS T4 ON T4.id_cliente = T3.id_cliente WHERE T4.genero = 'M' AND T3.tipo = 'TITULAR'"),
    (173, "SELECT T1.frequencia, T2.simbolo_k FROM conta AS T1 INNER JOIN (SELECT id_conta, simbolo_k, SUM(valor) AS total_valor FROM ordem GROUP BY id_conta, simbolo_k) AS T2 ON T1.id_conta = T2.id_conta WHERE T1.id_conta = 3 AND T2.total_valor = 3539"),
    (186, "SELECT CAST(SUM(T1.genero = 'M') AS REAL) * 100 / COUNT(T1.id_cliente) FROM cliente AS T1 INNER JOIN distrito AS T3 ON T1.id_distrito = T3.id_distrito INNER JOIN conta AS T2 ON T2.id_distrito = T3.id_distrito INNER JOIN disp as T4 on T1.id_cliente = T4.id_cliente AND T2.id_conta = T4.id_conta WHERE T2.frequencia = 'TAXA SEMANAL'"),
    (189, "SELECT T3.id_conta FROM cliente AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito INNER JOIN conta AS T3 ON T2.id_distrito = T3.id_distrito INNER JOIN disp AS T4 ON T1.id_cliente = T4.id_cliente AND T4.id_conta = T3.id_conta  WHERE T1.genero = 'F' ORDER BY T1.data_nascimento ASC, T2.A11 ASC LIMIT 1"),
    (192, "SELECT AVG(T2.valor) FROM conta AS T1 INNER JOIN emprestimo AS T2 ON T1.id_conta = T2.id_conta WHERE T2.status IN ('C', 'D') AND T1.frequencia = 'TAXA POR OPERACAO'"),
    (194, "SELECT T1.id_cliente, STRFTIME('%Y', CURRENT_TIMESTAMP) - STRFTIME('%Y', T3.data_nascimento) FROM disp AS T1 INNER JOIN cartao AS T2 ON T2.id_disposicao = T1.id_disposicao INNER JOIN cliente AS T3 ON T1.id_cliente = T3.id_cliente WHERE T2.tipo = 'gold' AND T1.tipo = 'TITULAR'"),
    (119, "SELECT T1.id_conta, T2.A2, T2.A3 FROM conta AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito WHERE T1.frequencia = 'TAXA POR OPERACAO' AND STRFTIME('%Y', T1.data)= '1993'"),
    (120, "SELECT T1.id_conta, T1.frequencia FROM conta AS T1 INNER JOIN distrito AS T2 ON T1.id_distrito = T2.id_distrito WHERE T2.A3 = 'east Bohemia' AND STRFTIME('%Y', T1.data) BETWEEN '1995' AND '2000'"),
]

for q_id, sql in queries:
    if q_id != 125:
        continue
    print(f"\n{'='*50}")
    print(f"Vamos rodar sql da questao {q_id}")
    try:
        df_result = pd.read_sql_query(sql, conn_test)
        print(f"Sucesso! Total de linhas: {len(df_result)}")
        if len(df_result) > 0:
            print("Primeiras 3 linhas:")
            display.display(df_result.head(3))
    except Exception as e:
        print(f"FALHA ao executar a query {q_id}:")
        print(e)



data/database.sqlite

Vamos rodar sql da questao 125
Sucesso! Total de linhas: 45
Primeiras 3 linhas:


,CAST((T3.A13 - T3.A12) AS REAL) * 100 / T3.A12
0,40.000000
1,39.259259
2,115.000000


In [8]:
df_result

,CAST((T3.A13 - T3.A12) AS REAL) * 100 / T3.A12
0,40.000000
1,39.259259
2,115.000000
3,22.500000
4,20.000000
5,NaN
6,11.666667
7,NaN
8,44.375000
9,13.333333


In [115]:
import json
import sqlite3
import pandas as pd
import IPython.display as display

dev_json_path = REPO_ROOT / "data" / "dev.json"

with dev_json_path.open("r", encoding="utf-8") as arquivo:
    perguntas_dev = json.load(arquivo)

conn_dev = sqlite3.connect(db_path_destino)

print(f"Arquivo carregado: {dev_json_path}")
print(f"Total de perguntas: {len(perguntas_dev)}")

try:
    for item in perguntas_dev:
        q_id = item.get("question_id")
        sql = item.get("sql") or item.get("SQL")

        print(f"\n{'=' * 50}")
        print(f"Rodando SQL da questao {q_id}")

        if not sql:
            print("FALHA: item sem campo sql/SQL.")
            continue

        try:
            df_resultado = pd.read_sql_query(sql, conn_dev)
            print(f"Sucesso! Total de linhas: {len(df_resultado)}")
            print("Primeiras 5 linhas:")
            display.display(df_resultado.head(5))
        except Exception as e:
            print(f"FALHA ao executar a query {q_id}:")
            print(e)
finally:
    conn_dev.close()


Arquivo carregado: /home/mileto/dev/text2sql-eval/data/dev.json
Total de perguntas: 32

Rodando SQL da questao 89
Sucesso! Total de linhas: 1
Primeiras 5 linhas:


,COUNT(T2.id_conta)
0,13



Rodando SQL da questao 92
Sucesso! Total de linhas: 1
Primeiras 5 linhas:


,COUNT(DISTINCT T2.id_distrito)
0,69



Rodando SQL da questao 93
Sucesso! Total de linhas: 1
Primeiras 5 linhas:


...

,id_cliente,"STRFTIME('%Y', CURRENT_TIMESTAMP) - STRFTIME('%Y', T3.data_nascimento)"
0,9,91
1,41,58
2,79,57
3,326,59
4,548,89



Rodando SQL da questao 119
Sucesso! Total de linhas: 21
Primeiras 5 linhas:


,id_conta,A2,A3
0,66,Rychnov nad Kneznou,east Bohemia
1,273,Karlovy Vary,west Bohemia
2,485,Kutna Hora,central Bohemia
3,539,Rakovnik,central Bohemia
4,1050,Hodonin,south Moravia



Rodando SQL da questao 120
Sucesso! Total de linhas: 364
Primeiras 5 linhas:


,id_conta,frequencia
0,14,TAXA MENSAL
1,76,TAXA MENSAL
2,80,TAXA MENSAL
3,84,TAXA POR OPERACAO
4,103,TAXA MENSAL


In [44]:
# Fechar conexão, caso ainda esteja aberta
if conn is not None:
    conn.close()
    conn = None
print("Conexão com o banco de dados fechada.")

Conexão com o banco de dados fechada.
